## Translate

In [7]:
# Prerequis : uv add transformers torch sentencepiece
# Necessite un acces reseau a huggingface.co (bloque dans le sandbox
# utilise pour ecrire ce code, a lancer chez toi)
# Utilise les fonctions reelles de src/multilingual/translation_based.py

# --- BLOC 1 : traduire une phrase, langue par langue ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from multilingual.translation import (
    TRANSLATION_MODELS,
    classify_via_translation,
    translate_to_english,
)

print("Langues supportees :", list(TRANSLATION_MODELS))

phrases_test = {
    "es": "La entrega fue muy lenta pero el producto es excelente",
    "de": "Die Lieferung war sehr langsam, aber das Produkt ist " "ausgezeichnet",
    "fr": "La livraison etait tres lente mais le produit est excellent",
    "hi": "डिलीवरी बहुत धीमी थी " "लेकिन उत्पाद उत्कृष्ट है",
}

print("\n=== Traductions vers l'anglais ===")
for lang, phrase in phrases_test.items():
    traduction = translate_to_english(phrase, lang)
    print(f"[{lang}] {phrase}")
    print(f"     -> {traduction}\n")

Langues supportees : ['es', 'de', 'fr', 'hi']

=== Traductions vers l'anglais ===


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[es] La entrega fue muy lenta pero el producto es excelente
     -> Delivery was very slow but the product is excellent



...

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[fr] La livraison etait tres lente mais le produit est excellent
     -> Delivery was very slow but the product is excellent



Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[hi] डिलीवरी बहुत धीमी थी लेकिन उत्पाद उत्कृष्ट है
     -> The delivery was very slow but the product is outstanding



In [4]:
# --- BLOC 2 : le cache -- verifier qu'un 2eme appel ne retelecharge pas ---
import time

debut = time.time()
translate_to_english("Otra frase de prueba", "es")
duree_premier_appel = time.time() - debut

debut = time.time()
translate_to_english("Una tercera frase", "es")
duree_deuxieme_appel = time.time() - debut

print(f"1er appel (modele deja charge dans ce script) : " f"{duree_premier_appel:.3f}s")
print(f"2eme appel (reutilise le cache) : {duree_deuxieme_appel:.3f}s")
print("-> le 2eme appel doit etre nettement plus rapide, le modele")
print("   MarianMT espagnol->anglais n'est charge qu'une seule fois")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1er appel (modele deja charge dans ce script) : 0.145s
2eme appel (reutilise le cache) : 0.072s
-> le 2eme appel doit etre nettement plus rapide, le modele
   MarianMT espagnol->anglais n'est charge qu'une seule fois


In [5]:
# --- BLOC 3 : pipeline complet -- traduction PUIS classification ---
from transformers import AutoModelForSequenceClassification, AutoTokenizer

REPO_MODELE_ENTRAINE = "Steeve2ml/globatrend-sentiment-distilbert"

tokenizer_en = AutoTokenizer.from_pretrained(REPO_MODELE_ENTRAINE)
modele_en = AutoModelForSequenceClassification.from_pretrained(REPO_MODELE_ENTRAINE)


print("\n=== Pipeline complet : traduction + classification ===")
for lang, phrase in phrases_test.items():
    resultat = classify_via_translation(phrase, lang, modele_en, tokenizer_en)
    print(f"[{lang}] {resultat['original_text']}")
    print(f"     traduit  : {resultat['translated_text']}")
    print(f"     predit   : {resultat['predicted_label']}")
    print(f"     confidence : {resultat['confidence']:.3f}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Pipeline complet : traduction + classification ===


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[es] La entrega fue muy lenta pero el producto es excelente
     traduit  : Delivery was very slow but the product is excellent
     predit   : negative
     confidence : 0.506



[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[de] Die Lieferung war sehr langsam, aber das Produkt ist ausgezeichnet
     traduit  : The delivery was very slow, but the product is excellent
     predit   : positive
     confidence : 0.502



[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[fr] La livraison etait tres lente mais le produit est excellent
     traduit  : Delivery was very slow but the product is excellent
     predit   : negative
     confidence : 0.506

[hi] डिलीवरी बहुत धीमी थी लेकिन उत्पाद उत्कृष्ट है
     traduit  : The delivery was very slow but the product is outstanding
     predit   : negative
     confidence : 0.506



In [6]:
# --- BLOC 4 : cas ou la traduction peut deformer le sens ---
# Teste des phrases avec une negation ou une ironie, plus difficiles a
# traduire correctement -- observe si la classification finale reste
# juste malgre une traduction imparfaite
phrases_difficiles = {
    "fr": "Ce n'est pas mauvais, mais je ne recommande pas vraiment",
    "es": "No esta mal, pero tampoco lo recomendaria realmente",
}

print("=== Cas difficiles (negation, nuance) ===")
for lang, phrase in phrases_difficiles.items():
    resultat = classify_via_translation(phrase, lang, modele_en, tokenizer_en)
    print(f"[{lang}] {resultat['original_text']}")
    print(f"     traduit : {resultat['translated_text']}")
    print(f"     predit  : {resultat['predicted_label']}")
    print(f"     confidence : {resultat['confidence']:.3f}")
    print("     (attendu : plutot negatif/mitige, verifie si la")
    print("      traduction a bien conserve la negation)\n")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Cas difficiles (negation, nuance) ===


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[fr] Ce n'est pas mauvais, mais je ne recommande pas vraiment
     traduit : It's not bad, but I don't really recommend it.
     predit  : negative
     confidence : 0.678
     (attendu : plutot negatif/mitige, verifie si la
      traduction a bien conserve la negation)

[es] No esta mal, pero tampoco lo recomendaria realmente
     traduit : Not bad, but I wouldn't really recommend it either.
     predit  : negative
     confidence : 0.683
     (attendu : plutot negatif/mitige, verifie si la
      traduction a bien conserve la negation)



In [7]:
# --- BLOC 5 : langue non supportee -- verifier l'erreur explicite ---
try:
    translate_to_english("何か", "ja")  # japonais, pas dans TRANSLATION_MODELS
except ValueError as e:
    print("Erreur attendue pour une langue non supportee :", e)

Erreur attendue pour une langue non supportee : Langue 'ja' non supportee. Choix : ['es', 'de', 'fr', 'hi']


## Cross lingual

In [2]:
# --- BLOC 1 : charger XLM-R et verifier le multi-script ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from multilingual.cross_lingual import (
    compare_languages_zero_shot,
    evaluate_zero_shot,
    fine_tune_on_source_language,
    load_xlmr_classifier,
)

model, tokenizer = load_xlmr_classifier()
print("Parametres XLM-R :", sum(p.numel() for p in model.parameters()))

textes_multi_scripts = [
    "The delivery was slow",
    "La livraison etait lente",
    "Die Lieferung war langsam",
    "डिलीवरी बहुत धीमी थी",
]
for texte in textes_multi_scripts:
    print(f"  {texte!r:40} -> {tokenizer.tokenize(texte)[:6]}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parametres XLM-R : 278045186
  'The delivery was slow'                  -> ['▁The', '▁delivery', '▁was', '▁slow']
  'La livraison etait lente'               -> ['▁La', '▁livraison', '▁eta', 'it', '▁lente']
  'Die Lieferung war langsam'              -> ['▁Die', '▁Lieferung', '▁war', '▁langsam']
  'डिलीवरी बहुत धीमी थी'                   -> ['▁डि', 'ली', 'वरी', '▁बहुत', '▁', 'धी']


In [4]:
# --- BLOC 2 : fine-tuner UNIQUEMENT sur l'anglais ---
from classical_ml.data_loader import load_movie_reviews

X_train, X_test, y_train, y_test = load_movie_reviews()
y_train_num = [1 if lab == "pos" else 0 for lab in y_train]
y_test_num = [1 if lab == "pos" else 0 for lab in y_test]

trainer = fine_tune_on_source_language(
    model, tokenizer, X_train, y_train_num, X_test, y_test_num, epochs=3
)
resultats_anglais = evaluate_zero_shot(trainer, tokenizer, X_test, y_test_num)
print("\nAccuracy sur l'anglais (langue source) :", resultats_anglais)

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.687528,0.675000,0.671377
2,No log,0.579840,0.715000,0.701852
3,No log,0.547211,0.762500,0.761712


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.547211,3,0.762500,0.761712



Accuracy sur l'anglais (langue source) : {'eval_loss': 0.547211229801178, 'eval_accuracy': 0.7625, 'eval_f1': 0.76171216083175}


In [5]:
# --- BLOC 3 : evaluation ZERO-SHOT sur les 4 AUTRES langues ---
# Ces phrases sont traduites A LA MAIN (verification manuelle), pas
# un vrai dataset -- juste assez pour un premier signal. Pour une
# vraie evaluation, utiliser MARC (ES/DE/FR) et IndicSentiment (HI)
# comme prevu dans plan-projet-globatrend-insights.md.

jeux_de_test = {
    "es": (
        [
            "Este producto es absolutamente fantastico, lo recomiendo",
            "Entrega rapida y embalaje impecable",
            "Muy decepcionado con la calidad, producto roto al llegar",
            "Servicio al cliente horrible, no volvere a pedir nunca",
        ],
        [1, 1, 0, 0],
    ),
    "de": (
        [
            "Dieses Produkt ist absolut fantastisch, ich empfehle es",
            "Schnelle Lieferung und tadellose Verpackung",
            "Sehr enttaeuscht von der Qualitaet, Produkt kaputt angekommen",
            "Schrecklicher Kundenservice, ich werde nie wieder bestellen",
        ],
        [1, 1, 0, 0],
    ),
    "fr": (
        [
            "Ce produit est absolument fantastique, je recommande",
            "Livraison rapide et emballage impeccable",
            "Tres decu par la qualite, produit casse a la reception",
            "Service client horrible, je ne commanderai plus jamais",
        ],
        [1, 1, 0, 0],
    ),
    "hi": (
        [
            "यह उत्पाद बिल्कुल " "शानदार है, " "मैं सिफारिश करता हूं",
            "तेज़ डिलीवरी और " "बेहतरीन पैकेजिंग",
            "गुणवत्ता से बहुत निराश, " "उत्पाद टूटा हुआ पहुंचा",
            "भयानक ग्राहक सेवा, " "मैं फिर कभी " "ऑर्डर नहीं करूंगा",
        ],
        [1, 1, 0, 0],
    ),
}

resultats_multilingues = compare_languages_zero_shot(trainer, tokenizer, jeux_de_test)

print("\n{:12} {:>10}".format("Langue", "Accuracy"))
print("{:12} {:>10.3f}".format("EN (source)", resultats_anglais["eval_accuracy"]))
for lang, res in resultats_multilingues.items():
    print(f"{lang:12} {res['eval_accuracy']:>10.3f}")

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.608759,3,1.000000,1.000000


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.603824,3,1.000000,1.000000


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.590407,3,1.000000,1.000000


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.613565,3,1.000000,1.000000



Langue         Accuracy
EN (source)       0.762
es                1.000
de                1.000
fr                1.000
hi                1.000


In [8]:
# --- BLOC 4 : APPROCHE 1 -- traduction puis classification ---
from transformers import AutoModelForSequenceClassification, AutoTokenizer

REPO_MODELE_ENTRAINE = "Steeve2ml/globatrend-sentiment-distilbert"

tokenizer_en = AutoTokenizer.from_pretrained(REPO_MODELE_ENTRAINE)
modele_en = AutoModelForSequenceClassification.from_pretrained(REPO_MODELE_ENTRAINE)

print("\n=== Approche 1 : traduction + DistilBERT anglais ===")
for lang, (textes, labels) in jeux_de_test.items():
    for texte, label_attendu in zip(textes, labels):
        resultat = classify_via_translation(texte, lang, modele_en, tokenizer_en)
        print(f"  [{lang}] {texte[:40]!r}")
        print(f"       -> traduit : {resultat['translated_text']!r}")
        print(f"       -> predit : {resultat['predicted_label']}")
        print(f"       -> confidence : {resultat['confidence']:.3f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Approche 1 : traduction + DistilBERT anglais ===


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [es] 'Este producto es absolutamente fantastic'
       -> traduit : 'This product is absolutely fantastic, I recommend'
       -> predit : positive
       -> confidence : 0.595


...

  [hi] 'यह उत्पाद बिल्कुल शानदार है, मैं सिफारिश'
       -> traduit : 'This product is perfect, I recommend'
       -> predit : positive
       -> confidence : 0.514


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hi] 'तेज़ डिलीवरी और बेहतरीन पैकेजिंग'
       -> traduit : 'Fast delivery and Fine Packageping'
       -> predit : positive
       -> confidence : 0.517


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [hi] 'गुणवत्ता से बहुत निराश, उत्पाद टूटा हुआ '
       -> traduit : 'Most disappointed by quality, product broken'
       -> predit : negative
       -> confidence : 0.733
  [hi] 'भयानक ग्राहक सेवा, मैं फिर कभी ऑर्डर नही'
       -> traduit : "The terrible customer service, I'll never order again."
       -> predit : negative
       -> confidence : 0.797


In [9]:
# --- BLOC 5 : APPROCHE 2 -- fine-tuning multilingue simultane ---
from multilingual.cross_lingual import fine_tune_on_multiple_languages

# on reutilise les memes 4 phrases par langue, mais cette fois comme
# donnees d'ENTRAINEMENT (mélangees), pas seulement de test -- avec un
# vrai dataset (MARC/IndicSentiment), utiliser des centaines d'exemples
# par langue plutot que 4
model2, tokenizer2 = load_xlmr_classifier()

trainer_multi = fine_tune_on_multiple_languages(
    model2,
    tokenizer2,
    jeux_de_test,
    X_test[:20],
    y_test_num[:20],
    epochs=5,
)
resultats_multi = compare_languages_zero_shot(trainer_multi, tokenizer2, jeux_de_test)

print("\n=== Approche 2 : XLM-R fine-tune sur les 5 langues a la fois ===")
for lang, res in resultats_multi.items():
    print(f"  {lang} : {res['eval_accuracy']:.3f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-p

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.688333,0.600000,0.375000
2,No log,0.688052,0.600000,0.375000
3,No log,0.689885,0.600000,0.375000
4,No log,0.690865,0.600000,0.375000
5,No log,0.690998,0.600000,0.375000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.692946,5,0.500000,0.333333


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.694966,5,0.500000,0.333333



=== Approche 2 : XLM-R fine-tune sur les 5 langues a la fois ===
  es : 0.500
  de : 0.500
  fr : 0.500
  hi : 0.500


In [10]:
# --- BLOC 6 : COMPARAISON FINALE DES 3 APPROCHES ---
print("\n{:12} {:>15} {:>15}".format("Langue", "Zero-shot (2)", "Multi (5 langues)"))
for lang in ["es", "de", "fr", "hi"]:
    zs = resultats_multilingues[lang]["eval_accuracy"]
    multi = resultats_multi[lang]["eval_accuracy"]
    print(f"{lang:12} {zs:>15.3f} {multi:>15.3f}")


Langue         Zero-shot (2) Multi (5 langues)
es                     1.000           0.500
de                     1.000           0.500
fr                     1.000           0.500
hi                     1.000           0.500
